# 🏷️ SKU Product Scoring Model

This notebook builds a **composite product scoring model** that ranks every SKU across multiple performance dimensions:

| Dimension | Weight | Description |
|---|---|---|
| Revenue Contribution | 35% | Net revenue generated |
| Sales Velocity | 30% | Units sold per day |
| Margin Proxy | 20% | Avg unit price (as margin proxy) |
| Discount Sensitivity | 15% | Lower discount = higher score |

## Sections
1. Setup & Data Loading
2. Feature Engineering per SKU
3. Score Normalisation
4. Composite Score & Ranking
5. Product Tier Classification
6. Visualisation & Export

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

WEIGHTS = {
    'revenue_score':  0.35,
    'velocity_score': 0.30,
    'margin_score':   0.20,
    'discount_score': 0.15,
}

print('Config loaded ✓')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/sample_orders.csv', parse_dates=['order_date'])
print(f'Orders loaded: {len(df):,}')
df.head()

## 2. Feature Engineering per SKU

In [ ]:
# Date range for velocity calculation
date_range_days = (df['order_date'].max() - df['order_date'].min()).days + 1

# Gross/net revenue
df['gross_revenue'] = df['quantity'] * df['unit_price']
df['net_revenue']   = df['gross_revenue'] * (1 - df['discount'])

sku_df = df.groupby(['sku', 'category']).agg(
    total_orders    =('order_id',      'nunique'),
    units_sold      =('quantity',      'sum'),
    net_revenue     =('net_revenue',   'sum'),
    avg_unit_price  =('unit_price',    'mean'),
    avg_discount    =('discount',      'mean'),
).reset_index()

sku_df['units_per_day'] = sku_df['units_sold'] / date_range_days

print(f'Unique SKUs: {len(sku_df)}')
sku_df

## 3. Normalise & Score

In [ ]:
scaler = MinMaxScaler()

# Higher is better → scale 0-1
sku_df['revenue_score']  = scaler.fit_transform(sku_df[['net_revenue']])
sku_df['velocity_score'] = scaler.fit_transform(sku_df[['units_per_day']])
sku_df['margin_score']   = scaler.fit_transform(sku_df[['avg_unit_price']])

# Lower discount is better → invert
sku_df['discount_score'] = 1 - scaler.fit_transform(sku_df[['avg_discount']])

# Composite score
sku_df['composite_score'] = sum(
    sku_df[col] * weight for col, weight in WEIGHTS.items()
)

sku_df = sku_df.sort_values('composite_score', ascending=False).reset_index(drop=True)
sku_df['rank'] = sku_df.index + 1

sku_df[['sku', 'category', 'composite_score', 'rank']].head(10)

## 4. Tier Classification

In [ ]:
def assign_tier(score: float) -> str:
    if score >= 0.75: return 'A — Star'
    elif score >= 0.50: return 'B — Performer'
    elif score >= 0.25: return 'C — Developing'
    else: return 'D — Review'

sku_df['tier'] = sku_df['composite_score'].apply(assign_tier)

tier_counts = sku_df['tier'].value_counts()
print('Product Tier Distribution:')
print(tier_counts.to_string())

sku_df[['rank', 'sku', 'category', 'net_revenue', 'units_sold',
        'avg_discount', 'composite_score', 'tier']].round(3)

## 5. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: revenue vs velocity coloured by tier
tier_colours = {
    'A — Star': '#2ecc71',
    'B — Performer': '#3498db',
    'C — Developing': '#f39c12',
    'D — Review': '#e74c3c',
}
for tier, colour in tier_colours.items():
    mask = sku_df['tier'] == tier
    axes[0].scatter(
        sku_df.loc[mask, 'units_per_day'],
        sku_df.loc[mask, 'net_revenue'],
        label=tier, color=colour, s=120, alpha=0.8
    )
    for _, row in sku_df[mask].iterrows():
        axes[0].annotate(row['sku'], (row['units_per_day'], row['net_revenue']),
                         fontsize=7, alpha=0.7)

axes[0].set_xlabel('Units Sold per Day (Velocity)')
axes[0].set_ylabel('Net Revenue ($)')
axes[0].set_title('SKU Performance Matrix', fontsize=14, fontweight='bold')
axes[0].legend()

# Horizontal bar chart: composite score
axes[1].barh(
    sku_df['sku'],
    sku_df['composite_score'],
    color=[tier_colours[t] for t in sku_df['tier']]
)
axes[1].axvline(x=0.5, color='gray', linestyle='--', linewidth=1, label='Mid-point')
axes[1].set_xlabel('Composite Score (0–1)')
axes[1].set_title('SKU Composite Scores', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../dashboards/screenshots/sku_scoring.png', dpi=150, bbox_inches='tight')
plt.show()

# Export
sku_df.round(4).to_csv('../data/sku_scores.csv', index=False)
print('Output saved → ../data/sku_scores.csv ✓')